<a href="https://colab.research.google.com/github/Chetalam/cnn-skin-cancer-classification/blob/main/notebooks/04_data_splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Data Splitting

This notebook creates reproducible training, validation and test manifests.

PAD-UFES-20 images are grouped by patient. HAM10000 images are grouped
by lesion because patient identifiers are unavailable. Identical image
hashes are also kept in the same group.

The target proportions are 70% training, 15% validation and 15% testing.
Actual proportions may differ slightly because related images must
remain together.

The original images, cleaned manifest and database are not modified.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/cnn-skin-cancer-classification"
)

MANIFEST_PATH = (
    PROJECT_ROOT / "Datasets" / "Cleaned" / "clean_metadata.csv"
)
SPLIT_ROOT = PROJECT_ROOT / "Datasets" / "Splits"
REPORT_ROOT = PROJECT_ROOT / "Reports" / "Data_Splitting"

if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Cleaned manifest not found: {MANIFEST_PATH}"
    )

# Preserve identifiers as text, including patient and lesion IDs.
df = pd.read_csv(
    MANIFEST_PATH,
    dtype={
        "dataset": "string",
        "image_id": "string",
        "patient_id": "string",
        "lesion_id": "string",
        "image_hash": "string",
    },
)

required = {
    "dataset",
    "image_id",
    "patient_id",
    "lesion_id",
    "class_label",
    "image_path",
    "image_hash",
}

missing_columns = required - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

for column in [
    "dataset", "image_id", "patient_id",
    "lesion_id", "class_label", "image_path", "image_hash",
]:
    df[column] = df[column].astype("string").str.strip()
    df[column] = df[column].replace("", pd.NA)

CLASS_NAMES = ["BCC", "Melanoma", "SCC"]
CLASS_TO_INDEX = {
    name: index for index, name in enumerate(CLASS_NAMES)
}

if df["class_label"].isna().any():
    raise ValueError("Some records have missing class labels.")

if set(df["class_label"]) != set(CLASS_NAMES):
    raise ValueError(
        f"Unexpected classes: {df['class_label'].unique().tolist()}"
    )

if not set(df["dataset"]).issubset({"HAM10000", "PAD-UFES-20"}):
    raise ValueError("Unexpected or missing dataset names.")

for column in ["image_id", "image_path", "image_hash"]:
    if df[column].isna().any():
        raise ValueError(f"Missing values in {column}.")

if df.duplicated(["dataset", "image_id"]).any():
    raise ValueError("Repeated dataset/image identifiers found.")

df = df.reset_index(drop=True)

print("Clean records loaded:", len(df))
display(df.groupby(["dataset", "class_label"]).size().unstack(fill_value=0))

if len(df) != 2703:
    raise ValueError(
        "Expected 2,703 cleaned records. Check the manifest before proceeding."
    )

Mounted at /content/drive
Clean records loaded: 2703


class_label,BCC,Melanoma,SCC
dataset,,,
HAM10000,514,1113,0
PAD-UFES-20,839,52,185


In [2]:
parent = list(range(len(df)))


def find(index):
    while parent[index] != index:
        parent[index] = parent[parent[index]]
        index = parent[index]
    return index


def union(first, second):
    first_root = find(first)
    second_root = find(second)

    if first_root != second_root:
        parent[second_root] = first_root


seen_keys = {}

for index, row in df.iterrows():
    dataset = row["dataset"]
    keys = []

    if dataset == "PAD-UFES-20":
        if pd.isna(row["patient_id"]):
            raise ValueError(
                f"PAD patient ID missing for {row['image_id']}."
            )

        keys.append(
            ("patient", dataset, str(row["patient_id"]))
        )

    elif dataset == "HAM10000":
        if pd.isna(row["lesion_id"]):
            raise ValueError(
                f"HAM lesion ID missing for {row['image_id']}."
            )

    # Also link images of the same lesion where identifiers exist.
    if pd.notna(row["lesion_id"]):
        keys.append(
            ("lesion", dataset, str(row["lesion_id"]))
        )

    # Hash grouping applies across both datasets.
    keys.append(("hash", str(row["image_hash"])))

    for key in keys:
        if key in seen_keys:
            union(index, seen_keys[key])
        else:
            seen_keys[key] = index

roots = [find(index) for index in range(len(df))]
unique_roots = sorted(set(roots))

root_to_group = {
    root: f"group_{number:05d}"
    for number, root in enumerate(unique_roots)
}

df["group_id"] = [root_to_group[root] for root in roots]
df["label_index"] = df["class_label"].map(CLASS_TO_INDEX)

hash_label_counts = df.groupby("image_hash")["class_label"].nunique()

if (hash_label_counts > 1).any():
    raise ValueError(
        "Conflicting labels remain for identical image hashes. "
        "Return to the cleaning notebook."
    )

print("Independent image groups:", df["group_id"].nunique())
print("Largest group:", df.groupby("group_id").size().max())
print("Class order:", CLASS_TO_INDEX)

Independent image groups: 1581
Largest group: 10
Class order: {'BCC': 0, 'Melanoma': 1, 'SCC': 2}


In [3]:
SEED = 42
SEARCH_ATTEMPTS = 10_000

SPLIT_NAMES = np.array(["train", "validation", "test"])
TARGET_RATIOS = np.array([0.70, 0.15, 0.15])

# Balance each available dataset/class combination.
strata = sorted(
    set(zip(df["dataset"], df["class_label"]))
)
stratum_to_index = {
    value: index for index, value in enumerate(strata)
}

group_names = sorted(df["group_id"].unique())
group_to_index = {
    value: index for index, value in enumerate(group_names)
}

group_counts = np.zeros(
    (len(group_names), len(strata)),
    dtype=np.int64,
)

for row in df.itertuples(index=False):
    group_counts[
        group_to_index[row.group_id],
        stratum_to_index[(row.dataset, row.class_label)],
    ] += 1

stratum_totals = group_counts.sum(axis=0)

# Project dataset/class counts into the three cancer classes.
stratum_to_class = np.zeros(
    (len(strata), len(CLASS_NAMES)),
    dtype=np.int64,
)

for index, (_, class_name) in enumerate(strata):
    stratum_to_class[index, CLASS_TO_INDEX[class_name]] = 1

rng = np.random.default_rng(SEED)
best_assignment = None
best_score = np.inf

for attempt in range(SEARCH_ATTEMPTS):
    assignment = rng.choice(
        3,
        size=len(group_names),
        p=TARGET_RATIOS,
    )

    counts = np.zeros(
        (3, len(strata)),
        dtype=np.int64,
    )
    np.add.at(counts, assignment, group_counts)

    class_counts = counts @ stratum_to_class

    # Every split must contain all three classes.
    if (class_counts == 0).any():
        continue

    observed_ratios = counts / stratum_totals[None, :]
    overall_ratios = counts.sum(axis=1) / len(df)

    score = (
        np.mean(
            (observed_ratios - TARGET_RATIOS[:, None]) ** 2
        )
        + np.mean(
            (overall_ratios - TARGET_RATIOS) ** 2
        )
    )

    if score < best_score:
        best_score = score
        best_assignment = assignment.copy()

if best_assignment is None:
    raise ValueError(
        "No valid grouped split found. Inspect the available groups."
    )

group_to_split = {
    group: str(SPLIT_NAMES[best_assignment[index]])
    for index, group in enumerate(group_names)
}

df["split"] = df["group_id"].map(group_to_split)

split_summary = (
    df.groupby("split")
    .size()
    .reindex(SPLIT_NAMES)
    .rename("image_count")
    .to_frame()
)

split_summary["actual_percent"] = (
    split_summary["image_count"] / len(df) * 100
).round(2)

split_summary["target_percent"] = TARGET_RATIOS * 100

print("Split sizes:")
display(split_summary)

print("Class counts:")
display(
    pd.crosstab(df["split"], df["class_label"])
    .reindex(index=SPLIT_NAMES, columns=CLASS_NAMES, fill_value=0)
)

print("Dataset/class counts:")
display(
    df.groupby(["split", "dataset", "class_label"])
    .size()
    .rename("image_count")
    .reset_index()
)

Split sizes:


,image_count,actual_percent,target_percent
split,,,
train,1905,70.48,70.0
validation,390,14.43,15.0
test,408,15.09,15.0


Class counts:


class_label,BCC,Melanoma,SCC
split,,,
train,956,819,130
validation,200,164,26
test,197,182,29


Dataset/class counts:


,split,dataset,class_label,image_count
0,test,HAM10000,BCC,72
1,test,HAM10000,Melanoma,173
2,test,PAD-UFES-20,BCC,125
3,test,PAD-UFES-20,Melanoma,9
4,test,PAD-UFES-20,SCC,29
5,train,HAM10000,BCC,366
6,train,HAM10000,Melanoma,783
7,train,PAD-UFES-20,BCC,590
8,train,PAD-UFES-20,Melanoma,36
9,train,PAD-UFES-20,SCC,130


In [4]:
# Connected groups cannot cross split boundaries.
assert df.groupby("group_id")["split"].nunique().max() == 1

# Identical images cannot cross split boundaries.
assert df.groupby("image_hash")["split"].nunique().max() == 1

# PAD patients cannot cross split boundaries.
pad = df[df["dataset"] == "PAD-UFES-20"]
assert pad.groupby("patient_id")["split"].nunique().max() == 1

# Known lesions cannot cross split boundaries.
lesions = df.dropna(subset=["lesion_id"])
assert (
    lesions.groupby(["dataset", "lesion_id"])["split"]
    .nunique()
    .max()
    == 1
)

assert len(df) == 2703
assert df["split"].notna().all()

for split_name in SPLIT_NAMES:
    selected = df[df["split"] == split_name]
    assert set(selected["class_label"]) == set(CLASS_NAMES)

missing_paths = [
    path
    for path in df["image_path"]
    if not Path(str(path)).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        f"{len(missing_paths)} selected images are missing. "
        f"Examples: {missing_paths[:3]}"
    )

SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

ordered = df.sort_values(
    ["split", "dataset", "image_id"]
).reset_index(drop=True)

ordered.to_csv(
    SPLIT_ROOT / "split_metadata.csv",
    index=False,
)

for split_name in SPLIT_NAMES:
    ordered[ordered["split"] == split_name].to_csv(
        SPLIT_ROOT / f"{split_name}_metadata.csv",
        index=False,
    )

split_summary.to_csv(
    REPORT_ROOT / "split_summary.csv"
)

class_counts = (
    pd.crosstab(df["split"], df["class_label"])
    .reindex(index=SPLIT_NAMES, columns=CLASS_NAMES, fill_value=0)
)
class_counts.to_csv(REPORT_ROOT / "class_by_split.csv")

dataset_class_counts = (
    df.groupby(["split", "dataset", "class_label"])
    .size()
    .rename("image_count")
    .reset_index()
)
dataset_class_counts.to_csv(
    REPORT_ROOT / "dataset_class_by_split.csv",
    index=False,
)

config = {
    "seed": SEED,
    "search_attempts": SEARCH_ATTEMPTS,
    "target_ratios": {
        name: float(ratio)
        for name, ratio in zip(SPLIT_NAMES, TARGET_RATIOS)
    },
    "class_names": CLASS_NAMES,
    "class_to_index": CLASS_TO_INDEX,
    "record_count": len(df),
    "group_count": int(df["group_id"].nunique()),
    "grouping": {
        "PAD-UFES-20": "patient and available lesion identifiers",
        "HAM10000": "lesion identifiers",
        "both_datasets": "identical image hashes",
    },
    "source_manifest": str(MANIFEST_PATH),
}

(SPLIT_ROOT / "split_config.json").write_text(
    json.dumps(config, indent=2),
    encoding="utf-8",
)

print("DATA SPLITTING COMPLETE")
print("Total records:", len(df))
print("Missing image files:", len(missing_paths))
print("Group/patient/lesion/hash leakage checks: passed")
print("Manifests saved to:", SPLIT_ROOT)
print("Reports saved to:", REPORT_ROOT)

display(split_summary)
display(class_counts)

DATA SPLITTING COMPLETE
Total records: 2703
Missing image files: 0
Group/patient/lesion/hash leakage checks: passed
Manifests saved to: /content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/Splits
Reports saved to: /content/drive/MyDrive/cnn-skin-cancer-classification/Reports/Data_Splitting


,image_count,actual_percent,target_percent
split,,,
train,1905,70.48,70.0
validation,390,14.43,15.0
test,408,15.09,15.0


class_label,BCC,Melanoma,SCC
split,,,
train,956,819,130
validation,200,164,26
test,197,182,29


## Interpretation and limitations

The split keeps PAD patients, HAM lesions and identical-image groups
within a single partition. HAM patient-level separation cannot be
verified because patient identifiers are unavailable.

The proportions are approximate because related images remain together.
SCC is present only in the PAD-UFES-20 selection, creating a potential
association between dataset source and class.

All three model architectures will use these same saved splits.
Model selection and tuning will use training and validation data.
The test set will be reserved for final evaluation.